# K13 · Evaluate all samples (step 6, CPU — uses no GPU quota)

Computes the metric tables (`scripts/evaluate.py`: SIFID, diversity, NIQE/MUSIQ/NIMA, patch statistics, copying,
structure) for every experiment it finds, several jobs in parallel. Results: `eval/<experiment>/<job>/per_image.csv`
and `per_sample.csv`, all in `cs787_k13_eval.zip`.

**Inputs to attach first** (*Add Input*):
- *Your Work* → the latest versions of **K10**, **K11**, **K12** (their `cs787_*.zip` are found automatically);
- the Colab results of step 2: download the folder `MyDrive/CS787/outputs/phase4_reproduction` from Google Drive
  (right-click → Download, gives a zip), then *Add Input* → *Upload* → *New Dataset* → drop the zip → Create.
Already-evaluated jobs are skipped, so you can rerun it after attaching more results.

**How to run (Kaggle):**
1. Right panel → *Session options*: **Accelerator = None (CPU)**, **Internet = On**.
2. Top right **Save Version** → *Save & Run All (Commit)* → Save. It runs in the background (you can close the tab);
   watch it under *View Active Events* / the version's **Logs** (progress lines every 2 min).
3. When it finishes, the result is the version's **Output**: `cs787_k13_eval.zip`.
4. **Not finished?** (it stops itself before Kaggle's 12-h limit) → *Add Input* → *Your Work* → this notebook's
   latest version → then *Save Version* again. It continues where it stopped.

In [ ]:
import time
T_START = time.time()                      # session clock: work stops before SESSION_HOURS
NOTEBOOK = "K13_evaluate"
GITHUB_REPO   = "https://github.com/rishabh-vr/CS787-Single-Image-Diffusion-Editor.git"
GITHUB_BRANCH = "main"
SID_COMMIT    = "85381e3c6f29fc07144da854d86be07150404716"     # official code, audited commit
SINDDM_COMMIT = "e7107d9fd198928f26e278f99ec428e261726a45"  # SinDDM, audited commit
SESSION_HOURS = 11.0                       # Kaggle stops sessions at 12 h; we stop starting new work at this point

GROUPS      = "paper,patch,copy,structure"   # "patch,copy,structure" = fast pass without pretrained networks
MAX_SAMPLES = 50
SLOTS       = 2                              # jobs evaluated at the same time (Kaggle CPU sessions have 4 cores)
# experiment folder -> manifests that contain its input images
EVAL = {
    "phase4_reproduction": ["uncond_sinddm15"],
    "phase7_ood":          ["ood_setA", "synthetic_setB"],
    "phase8_ours":         ["sinddm_ood3", "lora8"],
    "phase8_sinddm":       ["uncond_sinddm15", "sinddm_ood3"],
    "phase8_lora":         ["lora8"],
    "phase9_ablations":    ["ablation_subset"],
    "phase9_ab6_ann":      ["ablation_subset"],
}

In [ ]:
import os, sys, subprocess, shutil, glob, json
if not os.path.isdir("/kaggle/working"):
    raise SystemExit("This notebook is written for Kaggle (kaggle.com/code).")
WORK, TMP = "/kaggle/working", "/tmp/cs787"
PROJECT = f"{TMP}/project"
shutil.rmtree(PROJECT, ignore_errors=True)
os.makedirs(TMP, exist_ok=True)
r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", GITHUB_BRANCH, GITHUB_REPO, PROJECT],
                   capture_output=True, text=True)
if r.returncode:
    raise SystemExit("Could not download the code from GitHub. Turn Internet ON: right panel -> Session options -> "
                     "Internet (needs a phone-verified Kaggle account), then run again.\n" + r.stderr)
sys.path.insert(0, f"{PROJECT}/src")
os.environ["PYTHONPATH"] = f"{PROJECT}/src" + os.pathsep + os.environ.get("PYTHONPATH", "")
from sidx.kaggle import gather_inputs, unpack_own, pack
from sidx.progress import Bars
print("code:", subprocess.run(["git", "-C", PROJECT, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
gather_inputs()          # results of earlier versions attached under /kaggle/input (resume)

In [ ]:
!pip install -q lpips pytorch-fid pyiqa opencv-python-headless scikit-image scipy pandas
!pip install -q --no-deps -e {PROJECT}
SID_ROOT = f"{TMP}/efficient-SID-code"
if not os.path.exists(SID_ROOT):
    subprocess.run(["git", "clone", "-q", "https://github.com/haojunqiu/efficient-SID-code", SID_ROOT], check=True)

In [ ]:
# natural images (balloons, forest, ...) are third-party and not in the repo: fetch + check sha256 against the manifests
r = subprocess.run([sys.executable, "scripts/prepare_natural.py", "--verify", "--sets", "uncond/sinddm15", "--sid-root", SID_ROOT],
                   cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-1200:], r.stderr[-1200:])
assert r.returncode == 0, "data check failed (see above)"

## Evaluate (parallel; one bar per experiment, counted in images)

In [ ]:
from sidx.gpupool import Task, run_pool
unpack_own()
OUT_ROOT, EVAL_ROOT = f"{WORK}/outputs", f"{WORK}/eval"
threads = max(1, (os.cpu_count() or 2) // SLOTS)
tasks, plan = [], []
for exp, mans in EVAL.items():
    for jj in sorted(glob.glob(f"{OUT_ROOT}/{exp}/*/job.json")):
        job_dir = os.path.dirname(jj); job = os.path.basename(job_dir)
        n_img = sum(1 for d in glob.glob(f"{job_dir}/*/") if glob.glob(f"{d}seed_*.png"))
        plan.append((exp, job, n_img))
        cmd = [sys.executable, "scripts/evaluate.py", f"{OUT_ROOT}/{exp}", "--only-job", job, "--groups", GROUPS,
               "--max-samples", str(MAX_SAMPLES), "--eval-root", EVAL_ROOT, "--no-progress"]
        for m in mans:
            cmd += ["--manifest", f"{PROJECT}/data/manifests/{m}.csv"]
        tasks.append(Task(f"{exp}:{job}", cmd, cwd=PROJECT, env={"OMP_NUM_THREADS": str(threads), "MKL_NUM_THREADS": str(threads)}))
print({e: sum(1 for p in plan if p[0] == e) for e in EVAL}, "jobs found")

def poll():
    res = {}
    for exp, job, n_img in plan:
        d = f"{EVAL_ROOT}/{exp}/{job}"
        try:
            done = json.load(open(f"{d}/progress.json"))["done"]
        except Exception:
            done = n_img if os.path.exists(f"{d}/stamp.json") else 0
        a, b = res.get(exp, (0, 0))
        res[exp] = (a + min(done, n_img), b + n_img)
    return res

try:
    run_pool(tasks, poll, slots=SLOTS, n_gpus=0, log_dir=f"{WORK}/logs/{NOTEBOOK}", unit="image",
             deadline=T_START + SESSION_HOURS * 3600 - 15 * 60)
finally:
    pack("k13_eval", kinds=("eval",), remove=False)

## Quick look: per-job means of the main metrics

In [ ]:
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200)
rows = []
for f in sorted(glob.glob(f"{EVAL_ROOT}/*/*/per_image.csv")):
    df = pd.read_csv(f)
    exp, job = f.split("/")[-3], f.split("/")[-2]
    rows.append({"experiment": exp, "job": job.split("__")[0], "images": len(df),
                 **df.select_dtypes("number").mean().to_dict()})
summary = pd.DataFrame(rows)
summary.to_csv(f"{WORK}/eval_summary.csv", index=False)
summary.round(4)

In [ ]:
os.makedirs(f"{WORK}/logs", exist_ok=True)
!pip freeze > {WORK}/logs/{NOTEBOOK}_pip_freeze.txt
print("finished; output files:", sorted(os.listdir(WORK)))